In [0]:
# Nombre: fx_rates_business_analysis.py
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# ========== CELDA 1: CONFIGURACIÓN ==========
catalog = "test_demo"
schema = "silver"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

# Configurar estilo de visualización
plt.style.use('seaborn-v0_8-darkgrid')

In [0]:
# ========== CELDA: ANÁLISIS DE ESTABILIDAD EN COP ==========
df_stability = spark.sql("""
    WITH stability_metrics_cop AS (
        SELECT 
            country,
            currency_to,
            COUNT(*) as dias_con_datos,
            -- Métricas calculadas sobre TODOS los días del período
            ROUND(AVG(avg_cop_per_unit), 2) as promedio_cop,
            ROUND(STDDEV(avg_cop_per_unit), 2) as desviacion_cop,
            ROUND(MIN(avg_cop_per_unit), 2) as minimo_cop,
            ROUND(MAX(avg_cop_per_unit), 2) as maximo_cop,
            -- CV calculado sobre todos los valores diarios
            ROUND((STDDEV(avg_cop_per_unit) / AVG(avg_cop_per_unit)) * 100, 2) as coeficiente_variacion_cop,
            -- Volatilidad promedio diaria (promedio del rango porcentual)
            ROUND(AVG(COALESCE(range_pct_cop, 0)), 2) as volatilidad_cop_promedio
        FROM test_demo.silver.fx_rates_daily
        WHERE extraction_date >= DATE_SUB(CURRENT_DATE(), 365)
          AND avg_cop_per_unit IS NOT NULL
        GROUP BY country, currency_to
    )
    SELECT 
        country,
        currency_to,
        promedio_cop as COP_promedio_por_unidad,
        COALESCE(coeficiente_variacion_cop, 0) as cv_pct,
        COALESCE(volatilidad_cop_promedio, 0) as volatilidad_cop_promedio,
        dias_con_datos,
        RANK() OVER (ORDER BY COALESCE(coeficiente_variacion_cop, 999) ASC) as ranking_estabilidad,
        CASE 
            WHEN coeficiente_variacion_cop IS NULL THEN '❓ Sin datos suficientes'
            WHEN coeficiente_variacion_cop < 5 THEN '✅ Muy Estable'
            WHEN coeficiente_variacion_cop < 10 THEN '🟡 Estable'
            WHEN coeficiente_variacion_cop < 20 THEN '⚠️ Volátil'
            ELSE '🔴 Muy Volátil'
        END as clasificacion_estabilidad
    FROM stability_metrics_cop
    ORDER BY COALESCE(coeficiente_variacion_cop, 999) ASC
""")

display(df_stability)

# Visualización
import plotly.graph_objects as go
fig1 = go.Figure()
df_stability_pd = df_stability.toPandas()

# Verificar que no haya valores nulos
df_stability_pd['cv_pct'] = df_stability_pd['cv_pct'].fillna(0)

fig1.add_trace(go.Bar(
    name='Coeficiente de Variación en COP (%)',
    x=df_stability_pd['country'],
    y=df_stability_pd['cv_pct'],
    text=df_stability_pd['cv_pct'].round(2),
    textposition='auto',
    marker_color=['green' if x < 10 else 'yellow' if x < 20 else 'red' 
                  for x in df_stability_pd['cv_pct']],
    hovertemplate='<b>%{x}</b><br>' +
                  'CV: %{y:.2f}%<br>' +
                  'Promedio COP: %{customdata:.2f}<br>',
    customdata=df_stability_pd['COP_promedio_por_unidad']
))

fig1.update_layout(
    title='Estabilidad de Monedas en COP (Menor CV = Mayor Estabilidad)',
    xaxis_title='País',
    yaxis_title='Coeficiente de Variación COP (%)',
    template='plotly_white',
    showlegend=False,
    height=500,
    hovermode='x unified'
)

fig1.show()

print("\n📊 RANKING DE ESTABILIDAD (en COP):")
print("="*70)
for _, row in df_stability_pd.iterrows():
    print(f"{row['ranking_estabilidad']}. {row['country']} ({row['currency_to']})")
    print(f"   - COP promedio por unidad: {row['COP_promedio_por_unidad']:.2f}")
    print(f"   - CV en COP: {row['cv_pct']:.2f}%")
    print(f"   - Volatilidad promedio: {row['volatilidad_cop_promedio']:.2f}%")
    print(f"   - Clasificación: {row['clasificacion_estabilidad']}")
    print("-"*70)

# Tabla resumen formateada
display(df_stability_pd[['ranking_estabilidad', 'country', 'COP_promedio_por_unidad', 
                         'cv_pct', 'volatilidad_cop_promedio', 'clasificacion_estabilidad']])



country,currency_to,COP_promedio_por_unidad,cv_pct,volatilidad_cop_promedio,dias_con_datos,ranking_estabilidad,clasificacion_estabilidad
Brazil,BRL,813.44,4.37,0.0,327,1,✅ Muy Estable
Mexico,MXN,209.08,5.93,0.0,329,2,🟡 Estable
Venezuela,VES,24.82,6.81,0.0,334,3,🟡 Estable
Argentina,ARS,2.62,25.55,0.0,332,4,🔴 Muy Volátil



📊 RANKING DE ESTABILIDAD (en COP):
1. Brazil (BRL)
   - COP promedio por unidad: 813.44
   - CV en COP: 4.37%
   - Volatilidad promedio: 0.00%
   - Clasificación: ✅ Muy Estable
----------------------------------------------------------------------
2. Mexico (MXN)
   - COP promedio por unidad: 209.08
   - CV en COP: 5.93%
   - Volatilidad promedio: 0.00%
   - Clasificación: 🟡 Estable
----------------------------------------------------------------------
3. Venezuela (VES)
   - COP promedio por unidad: 24.82
   - CV en COP: 6.81%
   - Volatilidad promedio: 0.00%
   - Clasificación: 🟡 Estable
----------------------------------------------------------------------
4. Argentina (ARS)
   - COP promedio por unidad: 2.62
   - CV en COP: 25.55%
   - Volatilidad promedio: 0.00%
   - Clasificación: 🔴 Muy Volátil
----------------------------------------------------------------------


ranking_estabilidad,country,COP_promedio_por_unidad,cv_pct,volatilidad_cop_promedio,clasificacion_estabilidad
1,Brazil,813.44,4.37,0.0,✅ Muy Estable
2,Mexico,209.08,5.93,0.0,🟡 Estable
3,Venezuela,24.82,6.81,0.0,🟡 Estable
4,Argentina,2.62,25.55,0.0,🔴 Muy Volátil


In [0]:
# ========== CELDA 4: CLASIFICACIÓN DE RIESGO ==========
risk_df = spark.sql( """
  

WITH base_cop AS (
    SELECT
        country,
        currency_to,
        extraction_date,
        avg_cop_per_unit,
        LAG(avg_cop_per_unit) OVER (
            PARTITION BY country, currency_to
            ORDER BY extraction_date
        ) AS prev_cop
    FROM test_demo.silver.fx_rates_daily
    WHERE extraction_date >= DATE_SUB(CURRENT_DATE(), 365)
      AND avg_cop_per_unit IS NOT NULL
),
daily_moves_cop AS (
    -- Cambio porcentual interdiario en COP
    SELECT
        country,
        currency_to,
        extraction_date,
        avg_cop_per_unit,
        prev_cop,
        CASE
            WHEN prev_cop IS NULL OR prev_cop = 0 THEN NULL
            ELSE ((avg_cop_per_unit - prev_cop) / prev_cop) * 100.0
        END AS cambio_diario_cop_pct
    FROM base_cop
),
risk_components_cop AS (
    SELECT
        country,
        currency_to,
        -- 1) Volatilidad histórica (CV anual en COP)
        ROUND((STDDEV(avg_cop_per_unit) / AVG(avg_cop_per_unit)) * 100, 2) AS cv_cop_anual,

        -- 2) Máx. cambio diario (interdiario en COP)
        ROUND(MAX(ABS(cambio_diario_cop_pct)), 2) AS max_cambio_diario_cop_pct,

        -- 3) % de días extremos (|cambio interdiario COP| > 3%)
        SUM(CASE WHEN ABS(cambio_diario_cop_pct) > 3 THEN 1 ELSE 0 END) AS dias_extremos_cop,
        COUNT(cambio_diario_cop_pct) AS total_dias_validos,
        ROUND(
            CASE
                WHEN COUNT(cambio_diario_cop_pct) = 0 THEN 0
                ELSE (SUM(CASE WHEN ABS(cambio_diario_cop_pct) > 3 THEN 1 ELSE 0 END) * 100.0)
                     / COUNT(cambio_diario_cop_pct)
            END, 2
        ) AS pct_dias_extremos_cop
    FROM daily_moves_cop
    GROUP BY country, currency_to
),
trend_calc_cop AS (
    -- Tendencia anual en COP (primero vs último del año)
    SELECT
        country,
        currency_to,
        FIRST_VALUE(avg_cop_per_unit) OVER (
            PARTITION BY country, currency_to
            ORDER BY extraction_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS first_cop_year,
        LAST_VALUE(avg_cop_per_unit) OVER (
            PARTITION BY country, currency_to
            ORDER BY extraction_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS last_cop_year
    FROM base_cop
),
trend_summary_cop AS (
    SELECT
        country,
        currency_to,
        MAX(first_cop_year) AS first_cop,
        MAX(last_cop_year) AS last_cop,
        ROUND(((MAX(last_cop_year) - MAX(first_cop_year)) / MAX(first_cop_year)) * 100, 2) AS cambio_anual_cop_pct
    FROM trend_calc_cop
    GROUP BY country, currency_to
),
risk_scoring_cop AS (
    SELECT
        rc.country,
        rc.currency_to,
        rc.cv_cop_anual,
        rc.max_cambio_diario_cop_pct,
        ABS(ts.cambio_anual_cop_pct) AS cambio_anual_cop_abs,
        rc.pct_dias_extremos_cop,

        -- Score de riesgo basado en métricas COP (0-100)
        (
            -- 40%: Volatilidad en COP (CV)
            CASE
                WHEN rc.cv_cop_anual < 5 THEN 0
                WHEN rc.cv_cop_anual < 10 THEN 10
                WHEN rc.cv_cop_anual < 20 THEN 25
                ELSE 40
            END +
            -- 30%: Máx. cambio diario en COP
            CASE
                WHEN rc.max_cambio_diario_cop_pct < 2 THEN 0
                WHEN rc.max_cambio_diario_cop_pct < 5 THEN 10
                WHEN rc.max_cambio_diario_cop_pct < 10 THEN 20
                ELSE 30
            END +
            -- 20%: Tendencia anual en COP
            CASE
                WHEN ABS(ts.cambio_anual_cop_pct) < 10 THEN 0
                WHEN ABS(ts.cambio_anual_cop_pct) < 30 THEN 10
                WHEN ABS(ts.cambio_anual_cop_pct) < 50 THEN 15
                ELSE 20
            END +
            -- 10%: Frecuencia de extremos en COP
            CASE
                WHEN rc.pct_dias_extremos_cop < 5 THEN 0
                WHEN rc.pct_dias_extremos_cop < 10 THEN 5
                ELSE 10
            END
        ) AS risk_score_cop
    FROM risk_components_cop rc
    LEFT JOIN trend_summary_cop ts
        ON rc.country = ts.country
        AND rc.currency_to = ts.currency_to
)
SELECT
    country,
    currency_to,
    risk_score_cop AS risk_score,
    cv_cop_anual AS volatilidad_cv,
    max_cambio_diario_cop_pct AS max_cambio_diario_pct,
    cambio_anual_cop_abs AS cambio_anual_pct,
    pct_dias_extremos_cop AS pct_dias_extremos,
    CASE
        WHEN risk_score_cop <= 25 THEN '🟢 RIESGO BAJO'
        WHEN risk_score_cop <= 50 THEN '🟡 RIESGO MEDIO'
        ELSE '🔴 RIESGO ALTO'
    END AS categoria_riesgo,
    CASE
        WHEN risk_score_cop <= 25 THEN 'Moneda estable en COP, movimientos predecibles'
        WHEN risk_score_cop <= 50 THEN 'Volatilidad moderada en COP, requiere monitoreo'
        ELSE 'Alta volatilidad en COP, considerar cobertura'
    END AS recomendacion
FROM risk_scoring_cop
ORDER BY risk_score_cop DESC;


""")

display(risk_df)

country,currency_to,risk_score,volatilidad_cv,max_cambio_diario_pct,cambio_anual_pct,pct_dias_extremos,categoria_riesgo,recomendacion
Argentina,ARS,80,25.55,56.7,8.71,12.69,🔴 RIESGO ALTO,"Alta volatilidad en COP, considerar cobertura"
Mexico,MXN,40,5.93,12.44,0.65,0.30,🟡 RIESGO MEDIO,"Volatilidad moderada en COP, requiere monitoreo"
Venezuela,VES,40,6.81,6.71,17.09,4.50,🟡 RIESGO MEDIO,"Volatilidad moderada en COP, requiere monitoreo"
Brazil,BRL,20,4.37,8.21,5.67,0.31,🟢 RIESGO BAJO,"Moneda estable en COP, movimientos predecibles"


In [0]:
# Imports
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import pandas as pd

# Pasar a pandas
risk_pd = risk_df.toPandas()

# Renombrar columnas para que coincidan con tu código de gráficos
risk_pd.rename(columns={
    'volatilidad_cv': 'Volatilidad_CV',
    'max_cambio_diario_pct': 'Max_Cambio_Diario',
    'pct_dias_extremos': 'Pct_Dias_Extremos'
}, inplace=True)

# Asegurar tipos numéricos
for c in ['Volatilidad_CV','Max_Cambio_Diario','Pct_Dias_Extremos','risk_score']:
    if c in risk_pd.columns:
        risk_pd[c] = pd.to_numeric(risk_pd[c], errors='coerce')

# Evitar NaN en gráficos
risk_pd[['Volatilidad_CV','Max_Cambio_Diario','Pct_Dias_Extremos','risk_score']] = \
    risk_pd[['Volatilidad_CV','Max_Cambio_Diario','Pct_Dias_Extremos','risk_score']].fillna(0)

# ---------------------------
# Gráfico de radar (por país)
# ---------------------------
categories = ['Volatilidad CV (COP)', 'Max Cambio Diario (COP)', '% Días Extremos', 'Score Total']  # Actualizado
max_cv = max(1.0, risk_pd['Volatilidad_CV'].max())

fig2 = go.Figure()
for _, row in risk_pd.iterrows():
    values = [
        row['Volatilidad_CV'],
        row['Max_Cambio_Diario'],
        row['Pct_Dias_Extremos'],
        row['risk_score'] / 100.0 * max_cv
    ]
    fig2.add_trace(go.Scatterpolar(
        r=values,
        theta=categories,
        fill='toself',
        name=f"{row['country']} ({row['currency_to']})"  # Más detalle
    ))

fig2.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, max_cv])),
    showlegend=True,
    title="Componentes de Riesgo por País (Métricas en COP)",  # Actualizado
    height=500
)
fig2.show()

# ---------------------------
# Matriz de riesgo (scatter)
# ---------------------------
fig3 = px.scatter(
    risk_pd,
    x='Volatilidad_CV',
    y='Max_Cambio_Diario',
    size='risk_score',
    color='country',
    hover_data=['Pct_Dias_Extremos','currency_to','risk_score','categoria_riesgo'],
    title='Matriz de Riesgo Cambiario (Basado en valores COP)',  # Actualizado
    labels={
        'Volatilidad_CV': 'Coeficiente de Variación COP (%)',  # Actualizado
        'Max_Cambio_Diario': 'Máximo Cambio Diario COP (%)'   # Actualizado
    },
    template='plotly_white'
)

# Zonas de riesgo (ajustadas para valores en COP)
fig3.add_shape(type="rect", x0=0, y0=0, x1=10, y1=5,  fillcolor="green",  opacity=0.2, layer="below")
fig3.add_shape(type="rect", x0=10, y0=5, x1=20, y1=10, fillcolor="yellow", opacity=0.2, layer="below")
fig3.add_shape(type="rect", x0=20, y0=10, x1=max(50, risk_pd['Volatilidad_CV'].max()*1.1), 
               y1=max(20, risk_pd['Max_Cambio_Diario'].max()*1.1), fillcolor="red", opacity=0.2, layer="below")

fig3.add_annotation(x=5,  y=2.5,  text="Bajo Riesgo (COP)",  showarrow=False)
fig3.add_annotation(x=15, y=7.5,  text="Riesgo Medio (COP)", showarrow=False)
fig3.add_annotation(x=30, y=15,   text="Alto Riesgo (COP)",  showarrow=False)

fig3.show()

In [0]:
import numpy as np# ============= CELDA 4: MATRIZ DE CORRELACIÓN - VISUALIZACIÓN =============


correlation_data = spark.sql( """
   WITH daily_rates_cop AS (
    SELECT 
        extraction_date,
        MAX(CASE WHEN country = 'Venezuela' THEN avg_cop_per_unit END) as Venezuela_COP,
        MAX(CASE WHEN country = 'Argentina' THEN avg_cop_per_unit END) as Argentina_COP,
        MAX(CASE WHEN country = 'Mexico' THEN avg_cop_per_unit END) as Mexico_COP,
        MAX(CASE WHEN country = 'Brazil' THEN avg_cop_per_unit END) as Brazil_COP
    FROM test_demo.silver.fx_rates_daily
    WHERE extraction_date >= DATE_SUB(CURRENT_DATE(), 365)
      AND avg_cop_per_unit IS NOT NULL
    GROUP BY extraction_date
)
SELECT * FROM daily_rates_cop
""").toPandas()

# Calcular matriz de correlación
corr_matrix = correlation_data[['Venezuela_COP', 'Argentina_COP', 'Mexico_COP', 'Brazil_COP']].corr()

# Crear heatmap interactivo
fig4 = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    text=np.round(corr_matrix.values, 2),
    texttemplate='%{text}',
    textfont={"size": 12},
    colorscale='RdBu_r',
    zmid=0,
    zmin=-1,
    zmax=1
))

fig4.update_layout(
    title='Matriz de Correlación entre Monedas',
    xaxis_title='País',
    yaxis_title='País',
    height=500,
    template='plotly_white'
)

fig4.show()

# Interpretación de correlaciones
print("\n📈 INTERPRETACIÓN DE CORRELACIONES:")
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_value = corr_matrix.iloc[i, j]
        país1 = corr_matrix.columns[i]
        país2 = corr_matrix.columns[j]
        
        if abs(corr_value) > 0.7:
            tipo = "ALTA"
            color = "🔴"
        elif abs(corr_value) > 0.4:
            tipo = "MEDIA"
            color = "🟡"
        else:
            tipo = "BAJA"
            color = "🟢"
            
        print(f"{color} {país1} - {país2}: {corr_value:.3f} ({tipo})")


📈 INTERPRETACIÓN DE CORRELACIONES:
🟡 Venezuela_COP - Argentina_COP: 0.696 (MEDIA)
🔴 Venezuela_COP - Mexico_COP: 0.776 (ALTA)
🔴 Venezuela_COP - Brazil_COP: -0.784 (ALTA)
🔴 Argentina_COP - Mexico_COP: 0.869 (ALTA)
🟡 Argentina_COP - Brazil_COP: -0.674 (MEDIA)
🔴 Mexico_COP - Brazil_COP: -0.743 (ALTA)


In [0]:
# ============= CELDA 5: DASHBOARD INTEGRADO EN COP =============

# Imports
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# --- 1) Asegurar pandas DataFrames ---
# Estabilidad (viene de df_stability Spark)
stability_pd = df_stability.toPandas() if hasattr(df_stability, 'toPandas') else df_stability
if 'cv_pct' not in stability_pd.columns and 'coeficiente_variacion_cop' in stability_pd.columns:
    stability_pd['cv_pct'] = pd.to_numeric(stability_pd['coeficiente_variacion_cop'], errors='coerce')

# Riesgo (viene de risk_df Spark)
risk_pd = risk_df.toPandas().copy()
risk_pd.rename(columns={
    'volatilidad_cv': 'Volatilidad_CV',
    'max_cambio_diario_pct': 'Max_Cambio_Diario',
    'pct_dias_extremos': 'Pct_Dias_Extremos'
}, inplace=True)
# Tipos numéricos
for c in ['Volatilidad_CV','Max_Cambio_Diario','Pct_Dias_Extremos','risk_score']:
    if c in risk_pd.columns:
        risk_pd[c] = pd.to_numeric(risk_pd[c], errors='coerce')

# Tendencia temporal en COP (consulta 30 días)
trend_pd = spark.sql("""
    SELECT extraction_date, country, avg_cop_per_unit
    FROM test_demo.silver.fx_rates_daily
    WHERE extraction_date >= DATE_SUB(CURRENT_DATE(), 30)
      AND avg_cop_per_unit IS NOT NULL
    ORDER BY extraction_date
""").toPandas()

# Asegurar tipos
trend_pd['extraction_date'] = pd.to_datetime(trend_pd['extraction_date'], errors='coerce')
trend_pd['avg_cop_per_unit'] = pd.to_numeric(trend_pd['avg_cop_per_unit'], errors='coerce')

# --- 2) Crear dashboard con subplots ---
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Estabilidad por País (CV en COP)', 
        'Tendencia Temporal (Valor en COP)', 
        'Clasificación de Riesgo (Score basado en COP)', 
        'Matriz de Riesgo (Volatilidad vs Cambio Máximo en COP)'
    ),
    specs=[[{'type': 'bar'}, {'type': 'scatter'}],
           [{'type': 'bar'}, {'type': 'scatter'}]],
    vertical_spacing=0.12, horizontal_spacing=0.1
)

# (1) Estabilidad: CV por país en COP
if not stability_pd.empty:
    fig.add_trace(
        go.Bar(
            x=stability_pd['country'],
            y=pd.to_numeric(stability_pd['cv_pct'], errors='coerce'),
            marker_color=['green' if x < 10 else 'yellow' if x < 20 else 'red' 
                         for x in stability_pd['cv_pct']],
            name='CV COP (%)',
            text=stability_pd['cv_pct'].round(2),
            textposition='auto'
        ),
        row=1, col=1
    )

# (2) Tendencia temporal en COP: líneas por país
for country in sorted(trend_pd['country'].dropna().unique()):
    country_data = trend_pd[trend_pd['country'] == country]
    fig.add_trace(
        go.Scatter(
            x=country_data['extraction_date'],
            y=country_data['avg_cop_per_unit'],
            name=str(country),
            mode='lines+markers',
            line=dict(width=2)
        ),
        row=1, col=2
    )

# (3) Clasificación de Riesgo: barras por país
if not risk_pd.empty:
    risk_colors = ['green' if x <= 25 else 'yellow' if x <= 50 else 'red'
                   for x in risk_pd['risk_score'].fillna(0)]
    fig.add_trace(
        go.Bar(
            x=risk_pd['country'],
            y=risk_pd['risk_score'],
            marker_color=risk_colors,
            name='Risk Score COP',
            text=risk_pd['risk_score'].round(0),
            textposition='auto'
        ),
        row=2, col=1
    )

# (4) Volatilidad vs Cambio Máximo (ambos en COP)
if not risk_pd.empty:
    fig.add_trace(
        go.Scatter(
            x=risk_pd['Volatilidad_CV'],
            y=risk_pd['Max_Cambio_Diario'],
            mode='markers+text',
            text=risk_pd['country'],
            textposition="top center",
            marker=dict(
                size=risk_pd['risk_score']/2,  # Tamaño proporcional al score
                color=risk_pd['risk_score'],
                colorscale='RdYlGn_r',
                showscale=True,
                colorbar=dict(title="Risk Score", x=1.1),
                line=dict(width=0.5, color='black')
            ),
            name='Risk Position COP'
        ),
        row=2, col=2
    )

# --- 3) Layout y ejes ---
fig.update_layout(
    height=850,
    showlegend=False,
    title_text="Dashboard de Análisis FX - Todas las Métricas en COP",
    template='plotly_white',
    margin=dict(l=40, r=20, t=80, b=40),
    font=dict(size=11)
)

# Actualizar títulos de ejes
fig.update_xaxes(title_text="País", row=1, col=1)
fig.update_yaxes(title_text="CV en COP (%)", row=1, col=1)

fig.update_xaxes(title_text="Fecha", row=1, col=2)
fig.update_yaxes(title_text="COP por unidad de moneda local", row=1, col=2)

fig.update_xaxes(title_text="País", row=2, col=1)
fig.update_yaxes(title_text="Score de Riesgo (0-100)", row=2, col=1)

fig.update_xaxes(title_text="CV en COP (%)", row=2, col=2)
fig.update_yaxes(title_text="Max Cambio Diario COP (%)", row=2, col=2)

# Añadir grid
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')

fig.show()

# --- 4) Resumen de insights ---
print("\n📊 INSIGHTS DEL DASHBOARD (Análisis en COP):")
print("="*70)

# Mejor y peor estabilidad
if not stability_pd.empty:
    best_stability = stability_pd.loc[stability_pd['cv_pct'].idxmin()]
    worst_stability = stability_pd.loc[stability_pd['cv_pct'].idxmax()]
    print(f"✅ Mayor estabilidad: {best_stability['country']} (CV: {best_stability['cv_pct']:.2f}%)")
    print(f"⚠️ Menor estabilidad: {worst_stability['country']} (CV: {worst_stability['cv_pct']:.2f}%)")

# Mayor y menor riesgo
if not risk_pd.empty:
    highest_risk = risk_pd.loc[risk_pd['risk_score'].idxmax()]
    lowest_risk = risk_pd.loc[risk_pd['risk_score'].idxmin()]
    print(f"\n🔴 Mayor riesgo: {highest_risk['country']} (Score: {highest_risk['risk_score']:.0f})")
    print(f"🟢 Menor riesgo: {lowest_risk['country']} (Score: {lowest_risk['risk_score']:.0f})")

# Tendencia promedio
if not trend_pd.empty:
    avg_cop_all = trend_pd.groupby('country')['avg_cop_per_unit'].mean()
    print(f"\n📈 Valores promedio COP por unidad (últimos 30 días):")
    for country, value in avg_cop_all.items():
        print(f"   {country}: {value:.2f} COP")

print("="*70)


📊 INSIGHTS DEL DASHBOARD (Análisis en COP):
✅ Mayor estabilidad: Brazil (CV: 4.37%)
⚠️ Menor estabilidad: Argentina (CV: 25.55%)

🔴 Mayor riesgo: Argentina (Score: 80)
🟢 Menor riesgo: Brazil (Score: 20)

📈 Valores promedio COP por unidad (últimos 30 días):
   Argentina: 2.11 COP
   Brazil: 787.10 COP
   Mexico: 201.35 COP
   Venezuela: 24.22 COP
